<a href="https://colab.research.google.com/github/rathodgayathri9-glitch/neural-networks-machine-learning-exercises/blob/main/Exercise-02-Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=============================================================
  Backpropagation Neural Network Prediction & Feature
  Analysis System — Breast Cancer Wisconsin (Diagnostic)
=============================================================
  Encoding  : Benign = 0  |  Malignant = 1
  Features  : 30 real-valued (all normalised before training)
  Samples   : 569  (357 Benign, 212 Malignant)
=============================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score
)
import time

# ─────────────────────────────────────────────
#  GLOBAL STATE  (loaded once, shared by all menus)
# ─────────────────────────────────────────────
data        = None
X_raw       = None
y           = None
feature_names = None
X_train_s   = None   # scaled
X_test_s    = None
y_train     = None
y_test      = None
scaler      = None

# Three model variants
bpnn_model  = None   # deep BPNN  (3 hidden layers)
mlp_model   = None   # standard MLP (2 hidden layers)
shlnn_model = None   # single hidden layer NN


# ─────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────
BENIGN    = 0
MALIGNANT = 1

def divider(char="─", width=60):
    print(char * width)

def header(title):
    divider("═")
    print(f"  {title}")
    divider("═")

def load_and_prepare():
    """Load dataset, encode labels, split & scale once."""
    global data, X_raw, y, feature_names
    global X_train_s, X_test_s, y_train, y_test, scaler

    raw = load_breast_cancer()
    X_raw = raw.data
    feature_names = raw.feature_names

    # sklearn encodes: 0 = malignant, 1 = benign  →  flip so Benign=0, Malignant=1
    y = np.where(raw.target == 1, BENIGN, MALIGNANT)

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    print(f"\n  Dataset loaded: {X_raw.shape[0]} samples, "
          f"{X_raw.shape[1]} features")
    print(f"  Benign (0): {np.sum(y==0)}  |  Malignant (1): {np.sum(y==1)}")
    print(f"  Train: {len(y_train)}  |  Test: {len(y_test)}")
    print(f"  All features normalised with StandardScaler\n")


def _ensure_bpnn():
    global bpnn_model
    if bpnn_model is None:
        print("  [Training BPNN …]")
        bpnn_model = MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            solver="adam",
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1
        )
        bpnn_model.fit(X_train_s, y_train)
        print("  BPNN ready.\n")


def _ensure_all_models():
    global mlp_model, shlnn_model
    _ensure_bpnn()
    if mlp_model is None:
        print("  [Training MLP …]")
        mlp_model = MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            max_iter=500,
            random_state=42,
            early_stopping=True
        )
        mlp_model.fit(X_train_s, y_train)
    if shlnn_model is None:
        print("  [Training SHLNN …]")
        shlnn_model = MLPClassifier(
            hidden_layer_sizes=(64,),
            activation="relu",
            solver="adam",
            max_iter=500,
            random_state=42,
            early_stopping=True
        )
        shlnn_model.fit(X_train_s, y_train)
    print("  All models ready.\n")


def _label(val):
    return "Malignant" if val == MALIGNANT else "Benign"


# ─────────────────────────────────────────────
#  MENU 1 — Tumor Classification using BPNN
# ─────────────────────────────────────────────
def menu_1_tumor_classification():
    header("1. Tumor Classification using BPNN")
    _ensure_bpnn()

    y_pred = bpnn_model.predict(X_test_s)

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, bpnn_model.predict_proba(X_test_s)[:, 1])

    print("  Confusion Matrix (rows = Actual, cols = Predicted):")
    print("             Benign(0)  Malignant(1)")
    print(f"  Benign(0)  {cm[0][0]:^9}  {cm[0][1]:^12}")
    print(f"  Malig.(1)  {cm[1][0]:^9}  {cm[1][1]:^12}\n")

    print(f"  Accuracy   : {acc*100:.2f}%")
    print(f"  ROC-AUC    : {auc:.4f}\n")

    print("  Classification Report:")
    divider()
    print(classification_report(
        y_test, y_pred,
        target_names=["Benign (0)", "Malignant (1)"]
    ))

    # Show first 10 predictions
    print("  Sample Predictions (first 10 test samples):")
    divider()
    print(f"  {'#':<4} {'Actual':<14} {'Predicted':<14} {'Correct?'}")
    divider()
    for i in range(10):
        actual = _label(y_test[i])
        pred   = _label(y_pred[i])
        ok     = "✓" if y_test[i] == y_pred[i] else "✗"
        print(f"  {i+1:<4} {actual:<14} {pred:<14} {ok}")
    divider()


# ─────────────────────────────────────────────
#  MENU 2 — Prediction Confidence Analyzer
# ─────────────────────────────────────────────
def menu_2_confidence_analyzer():
    header("2. Prediction Confidence Analyzer")
    _ensure_bpnn()

    proba  = bpnn_model.predict_proba(X_test_s)
    y_pred = bpnn_model.predict(X_test_s)

    conf_benign    = proba[:, 0]
    conf_malignant = proba[:, 1]

    # Confidence buckets
    buckets = {"90-100%": 0, "70-90%": 0, "50-70%": 0, "<50%*": 0}
    max_conf = np.max(proba, axis=1)
    for c in max_conf:
        if c >= 0.90: buckets["90-100%"] += 1
        elif c >= 0.70: buckets["70-90%"] += 1
        elif c >= 0.50: buckets["50-70%"] += 1
        else:           buckets["<50%*"]  += 1

    print("  Confidence Distribution across test set:")
    divider()
    for bucket, count in buckets.items():
        bar = "█" * (count // 2)
        print(f"  {bucket:<10}  {bar:<30} {count:>3} samples")
    print("  (* model uncertain; could go either way)\n")

    # Most confident correct & most confident wrong
    correct_mask = (y_pred == y_test)
    wrong_mask   = ~correct_mask

    if correct_mask.any():
        idx = np.argmax(max_conf * correct_mask.astype(float))
        print(f"  Most confident CORRECT prediction:")
        print(f"    Sample #{idx:>3} | Actual: {_label(y_test[idx]):<12}"
              f"| Pred: {_label(y_pred[idx]):<12}"
              f"| Confidence: {max_conf[idx]*100:.2f}%\n")

    if wrong_mask.any():
        idx = np.argmax(max_conf * wrong_mask.astype(float))
        print(f"  Most confident WRONG prediction (dangerous):")
        print(f"    Sample #{idx:>3} | Actual: {_label(y_test[idx]):<12}"
              f"| Pred: {_label(y_pred[idx]):<12}"
              f"| Confidence: {max_conf[idx]*100:.2f}%\n")

    # Top 10 by max confidence
    top10_idx = np.argsort(max_conf)[::-1][:10]
    print("  Top 10 Highest-Confidence Predictions:")
    divider()
    print(f"  {'#':<4} {'Actual':<14} {'Predicted':<14} "
          f"{'P(Benign)':<12} {'P(Malignant)':<14} {'Correct?'}")
    divider()
    for rank, i in enumerate(top10_idx, 1):
        ok = "✓" if y_test[i] == y_pred[i] else "✗"
        print(f"  {rank:<4} {_label(y_test[i]):<14} {_label(y_pred[i]):<14} "
              f"{conf_benign[i]:<12.4f} {conf_malignant[i]:<14.4f} {ok}")
    divider()


# ─────────────────────────────────────────────
#  MENU 3 — Feature Contribution Analyzer
# ─────────────────────────────────────────────
def menu_3_feature_contribution():
    header("3. Feature Contribution Analyzer")
    _ensure_bpnn()

    # Permutation importance (model-agnostic, works for BPNN)
    baseline_acc = accuracy_score(y_test, bpnn_model.predict(X_test_s))
    importances  = np.zeros(X_test_s.shape[1])

    print("  Computing permutation importance (this may take a moment)…\n")
    rng = np.random.RandomState(0)
    for col in range(X_test_s.shape[1]):
        X_perm = X_test_s.copy()
        X_perm[:, col] = rng.permutation(X_perm[:, col])
        drop = baseline_acc - accuracy_score(y_test, bpnn_model.predict(X_perm))
        importances[col] = drop

    ranked = np.argsort(importances)[::-1]

    print(f"  Baseline accuracy: {baseline_acc*100:.2f}%\n")
    print("  Feature importance (accuracy drop when feature is shuffled):")
    divider()
    print(f"  {'Rank':<6} {'Feature':<30} {'Importance':<12} {'Bar'}")
    divider()
    max_imp = importances[ranked[0]] if importances[ranked[0]] > 0 else 1
    for rank, idx in enumerate(ranked[:15], 1):
        imp = importances[idx]
        bar = "█" * max(1, int(imp / max_imp * 30)) if imp > 0 else ""
        print(f"  {rank:<6} {feature_names[idx]:<30} {imp:>+.4f}     {bar}")
    divider()
    print("\n  Positive = removing this feature hurts accuracy (important)")
    print("  Near zero / negative = feature contributes little\n")


# ─────────────────────────────────────────────
#  MENU 4 — Hidden Neuron Response Analyzer
# ─────────────────────────────────────────────
def menu_4_hidden_neuron_response():
    header("4. Hidden Neuron Response Analyzer")
    _ensure_bpnn()

    # Extract weights for each hidden layer
    n_layers = len(bpnn_model.coefs_)
    print(f"  BPNN Architecture: Input(30)")
    for i, coef in enumerate(bpnn_model.coefs_[:-1]):
        print(f"    → Hidden Layer {i+1}: {coef.shape[1]} neurons")
    print(f"    → Output: {bpnn_model.coefs_[-1].shape[1]} neuron(s)\n")

    # Show weight statistics per hidden layer
    print("  Weight Statistics per Hidden Layer:")
    divider()
    print(f"  {'Layer':<10} {'Neurons':<10} {'Min W':<10} "
          f"{'Max W':<10} {'Mean |W|':<12} {'Std W'}")
    divider()
    for i, coef in enumerate(bpnn_model.coefs_[:-1]):
        print(f"  {'Hidden '+str(i+1):<10} {coef.shape[1]:<10} "
              f"{coef.min():<10.4f} {coef.max():<10.4f} "
              f"{np.abs(coef).mean():<12.4f} {coef.std():.4f}")
    divider()

    # Simulate activation of hidden layer 1 for 5 samples
    # Manual forward pass through layer 1 (ReLU)
    W1 = bpnn_model.coefs_[0]
    b1 = bpnn_model.intercepts_[0]
    Z1 = X_test_s[:5] @ W1 + b1
    A1 = np.maximum(0, Z1)   # ReLU

    print(f"\n  Neuron Activations — Hidden Layer 1 (first 5 test samples):")
    print(f"  Showing top 8 neurons by mean activation:\n")
    mean_act = A1.mean(axis=0)
    top8     = np.argsort(mean_act)[::-1][:8]
    divider()
    print(f"  {'Neuron':<10}", end="")
    for s in range(5): print(f"  Sample{s+1:<6}", end="")
    print(f"  {'Mean':>8}")
    divider()
    for n in top8:
        print(f"  {'H1-N'+str(n):<10}", end="")
        for s in range(5): print(f"  {A1[s,n]:<12.4f}", end="")
        print(f"  {mean_act[n]:>8.4f}")
    divider()

    # Dead neuron analysis (neurons that are always 0 = dead ReLU)
    W1_full = bpnn_model.coefs_[0]
    b1_full = bpnn_model.intercepts_[0]
    Z_full  = X_test_s @ W1_full + b1_full
    A_full  = np.maximum(0, Z_full)
    dead    = np.sum(A_full.max(axis=0) == 0)
    print(f"\n  Dead neurons in Hidden Layer 1 (never activate): {dead}"
          f" / {W1_full.shape[1]}")


# ─────────────────────────────────────────────
#  MENU 5 — Input Feature Masking Challenge
# ─────────────────────────────────────────────
def menu_5_feature_masking():
    header("5. Input Feature Masking Challenge")
    _ensure_bpnn()

    baseline_acc = accuracy_score(y_test, bpnn_model.predict(X_test_s))
    print(f"  Baseline accuracy (all 30 features): {baseline_acc*100:.2f}%\n")

    # Mask each feature individually (set to 0 after scaling)
    print("  Accuracy when each feature is masked (zeroed out):")
    divider()
    print(f"  {'Feature':<30} {'Masked Acc':<14} {'Drop':<10} {'Impact'}")
    divider()

    results = []
    for col in range(X_test_s.shape[1]):
        X_masked = X_test_s.copy()
        X_masked[:, col] = 0.0
        acc   = accuracy_score(y_test, bpnn_model.predict(X_masked))
        drop  = baseline_acc - acc
        results.append((feature_names[col], acc, drop))

    results.sort(key=lambda x: x[2], reverse=True)

    for name, acc, drop in results[:30]:
        if   drop >  0.05: impact = "Critical"
        elif drop >  0.02: impact = "High"
        elif drop >  0.00: impact = "Moderate"
        elif drop == 0.00: impact = "Negligible"
        else:              impact = "Helps if removed"
        print(f"  {name:<30} {acc*100:<14.2f} {drop:>+.4f}    {impact}")
    divider()

    # Interactive: let user mask multiple features
    print("\n  Try masking multiple features together:")
    print("  (Enter feature numbers 1-30 separated by commas, or press Enter to skip)\n")
    for rank, (name, _, _) in enumerate(results[:30], 1):
        print(f"  {rank:>2}. {name}")

    try:
        raw_input = input("\n  Enter feature numbers to mask: ").strip()
        if raw_input:
            indices = [int(x.strip()) - 1 for x in raw_input.split(",")
                       if x.strip().isdigit()]
            indices = [i for i in indices if 0 <= i < 30]
            if indices:
                ranked_names = [r[0] for r in results]
                feat_indices = []
                for i in indices:
                    name = ranked_names[i]
                    feat_idx = list(feature_names).index(name)
                    feat_indices.append(feat_idx)

                X_masked = X_test_s.copy()
                for fi in feat_indices:
                    X_masked[:, fi] = 0.0
                acc_m = accuracy_score(y_test, bpnn_model.predict(X_masked))
                print(f"\n  Masking {len(feat_indices)} features:")
                for i in indices:
                    print(f"    - {ranked_names[i]}")
                print(f"\n  New accuracy: {acc_m*100:.2f}%  "
                      f"(drop: {(baseline_acc-acc_m)*100:.2f}%)")
    except (ValueError, KeyboardInterrupt):
        pass


# ─────────────────────────────────────────────
#  MENU 6 — Misclassification Confidence Analyzer
# ─────────────────────────────────────────────
def menu_6_misclassification():
    header("6. Misclassification Confidence Analyzer")
    _ensure_bpnn()

    proba  = bpnn_model.predict_proba(X_test_s)
    y_pred = bpnn_model.predict(X_test_s)
    max_conf = np.max(proba, axis=1)

    wrong_idx = np.where(y_pred != y_test)[0]

    print(f"  Total misclassifications: {len(wrong_idx)} / {len(y_test)}\n")

    if len(wrong_idx) == 0:
        print("  Perfect classification on test set — no errors to analyse.")
        return

    # Error type breakdown
    fn = np.sum((y_pred == BENIGN)    & (y_test == MALIGNANT))  # missed cancer
    fp = np.sum((y_pred == MALIGNANT) & (y_test == BENIGN))     # false alarm

    print(f"  False Negatives (missed cancer): {fn}  ← clinically dangerous")
    print(f"  False Positives (false alarm)  : {fp}\n")

    # Sort wrong by confidence (high-confidence errors are most dangerous)
    wrong_conf = max_conf[wrong_idx]
    sorted_w   = wrong_idx[np.argsort(wrong_conf)[::-1]]

    print("  Misclassified Samples (sorted by confidence, highest first):")
    divider()
    print(f"  {'#':<4} {'Idx':<6} {'Actual':<14} {'Predicted':<14} "
          f"{'Confidence':<12} {'Error Type'}")
    divider()
    for rank, i in enumerate(sorted_w, 1):
        etype = ("FN – Missed Cancer"
                 if y_test[i] == MALIGNANT and y_pred[i] == BENIGN
                 else "FP – False Alarm")
        print(f"  {rank:<4} {i:<6} {_label(y_test[i]):<14} "
              f"{_label(y_pred[i]):<14} "
              f"{max_conf[i]*100:<12.2f} {etype}")
    divider()

    if len(wrong_idx) > 0:
        most_wrong = sorted_w[0]
        print(f"\n  Most dangerous misclassification:")
        print(f"  Sample #{most_wrong} was ACTUALLY {_label(y_test[most_wrong])}"
              f" but predicted as {_label(y_pred[most_wrong])}"
              f" with {max_conf[most_wrong]*100:.2f}% confidence.")
        print(f"\n  P(Benign)={proba[most_wrong,0]:.4f}  "
              f"P(Malignant)={proba[most_wrong,1]:.4f}")

        print("\n  Top 5 deviating features vs test-set mean for this sample:")
        sample  = X_test_s[most_wrong]
        mean_ts = X_test_s.mean(axis=0)
        diff    = np.abs(sample - mean_ts)
        top5    = np.argsort(diff)[::-1][:5]
        for idx in top5:
            print(f"    {feature_names[idx]:<32} "
                  f"value={sample[idx]:+.4f}  mean={mean_ts[idx]:+.4f}"
                  f"  |diff|={diff[idx]:.4f}")


# ─────────────────────────────────────────────
#  MENU 7 — BPNN-MLP-SHLNN Comparative Analysis
# ─────────────────────────────────────────────
def menu_7_comparative():
    header("7. BPNN – MLP – SHLNN Comparative Analysis")
    _ensure_all_models()

    models = {
        "BPNN  (128-64-32)": bpnn_model,
        "MLP   (64-32)    ": mlp_model,
        "SHLNN (64)       ": shlnn_model,
    }

    rows = []
    for name, model in models.items():
        y_pred = model.predict(X_test_s)
        proba  = model.predict_proba(X_test_s)
        acc    = accuracy_score(y_test, y_pred)
        auc    = roc_auc_score(y_test, proba[:, 1])
        cm     = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        sens   = tp / (tp + fn) if (tp + fn) > 0 else 0  # recall for malignant
        spec   = tn / (tn + fp) if (tn + fp) > 0 else 0
        prec   = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1     = 2*prec*sens / (prec+sens) if (prec+sens) > 0 else 0
        iters  = model.n_iter_
        params = sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_)
        rows.append((name, acc, auc, sens, spec, prec, f1, iters, params))

    divider("═")
    print(f"  {'Model':<22} {'Acc':>7} {'AUC':>7} {'Sens':>7} "
          f"{'Spec':>7} {'Prec':>7} {'F1':>7} {'Iters':>7} {'Params':>8}")
    divider()
    for name, acc, auc, sens, spec, prec, f1, iters, params in rows:
        print(f"  {name:<22} {acc*100:>6.2f}% {auc:>7.4f} {sens:>7.4f} "
              f"{spec:>7.4f} {prec:>7.4f} {f1:>7.4f} {iters:>7} {params:>8}")
    divider("═")

    # Recommendation
    best_auc = max(rows, key=lambda r: r[2])
    best_sens = max(rows, key=lambda r: r[3])
    print(f"\n  Best AUC    : {best_auc[0].strip()}  ({best_auc[2]:.4f})")
    print(f"  Best Recall : {best_sens[0].strip()}  ({best_sens[3]:.4f})")
    print(f"\n  In a clinical setting, recall (sensitivity) matters most —")
    print(f"  missing cancer (FN) is costlier than a false alarm (FP).")

    # Per-model classification reports
    for name, model in models.items():
        print(f"\n  {'─'*20} {name.strip()} {'─'*20}")
        print(classification_report(
            y_test, model.predict(X_test_s),
            target_names=["Benign (0)", "Malignant (1)"]
        ))


# ─────────────────────────────────────────────
#  MAIN LOOP
# ─────────────────────────────────────────────
MENU_TEXT = """
  ╔══════════════════════════════════════════════════════╗
  ║   BPNN Breast Cancer Prediction & Analysis System   ║
  ╠══════════════════════════════════════════════════════╣
  ║  1. Tumor Classification using BPNN                 ║
  ║  2. Prediction Confidence Analyzer                  ║
  ║  3. Feature Contribution Analyzer                   ║
  ║  4. Hidden Neuron Response Analyzer                 ║
  ║  5. Input Feature Masking Challenge                 ║
  ║  6. Misclassification Confidence Analyzer           ║
  ║  7. BPNN-MLP-SHLNN Comparative Analysis            ║
  ║  8. Exit                                            ║
  ╚══════════════════════════════════════════════════════╝
"""

DISPATCH = {
    "1": menu_1_tumor_classification,
    "2": menu_2_confidence_analyzer,
    "3": menu_3_feature_contribution,
    "4": menu_4_hidden_neuron_response,
    "5": menu_5_feature_masking,
    "6": menu_6_misclassification,
    "7": menu_7_comparative,
}

def main():
    print("\n  Initialising…")
    load_and_prepare()

    while True:
        print(MENU_TEXT)
        choice = input("  Enter choice (1-8): ").strip()

        if choice == "8":
            print("\n  Exiting. Goodbye!\n")
            break
        elif choice in DISPATCH:
            print()
            try:
                DISPATCH[choice]()
            except Exception as e:
                print(f"\n  Error: {e}")
            input("\n  Press Enter to return to menu…")
        else:
            print("  Invalid choice. Please enter 1–8.\n")


if __name__ == "__main__":
    main()


  Initialising…

  Dataset loaded: 569 samples, 30 features
  Benign (0): 357  |  Malignant (1): 212
  Train: 455  |  Test: 114
  All features normalised with StandardScaler


  ╔══════════════════════════════════════════════════════╗
  ║   BPNN Breast Cancer Prediction & Analysis System   ║
  ╠══════════════════════════════════════════════════════╣
  ║  1. Tumor Classification using BPNN                 ║
  ║  2. Prediction Confidence Analyzer                  ║
  ║  3. Feature Contribution Analyzer                   ║
  ║  4. Hidden Neuron Response Analyzer                 ║
  ║  5. Input Feature Masking Challenge                 ║
  ║  6. Misclassification Confidence Analyzer           ║
  ║  7. BPNN-MLP-SHLNN Comparative Analysis            ║
  ║  8. Exit                                            ║
  ╚══════════════════════════════════════════════════════╝

  Enter choice (1-8): 1

════════════════════════════════════════════════════════════
  1. Tumor Classification using BPNN
═